# Exploration 12. Session-Based Recommendation


__Session-Based Recommendation__ ? ? ??? __?? ???? ???? ??? ??? ?? ?? ??? ???? ??__ ?? ????.
> Session ???  
??? ???? ????? ???? ??? ??? ?? ???? ???, ?? ?? ???

Session ? __????? ???? ??? ??? ??? ?? ??? ???__ ? ???? E-Commerce ?? ???? ?????.

![image](https://user-images.githubusercontent.com/84179578/130542301-8472e8a6-2fa9-4232-a0f6-70ad1547f23d.png)

## 0. ??? ?? ????

In [ ]:
import datetime as dt
from pathlib import Path
import os

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. ??? ??

2015??? YOOCHOOSE ?? ?? ?? ??? ???? ??? [E-Commerce ???](https://webcache.googleusercontent.com/search?q=cache:Npc6PQ3mzngJ:https://recsys.acm.org/recsys15/challenge/+&cd=1&hl=en&ct=clnk&gl=kr)? ????.

In [ ]:
# ??? ??
import os
f = open(os.getenv('HOME')+'/aiffel/yoochoose/data/dataset-README.txt', 'r')
while True:
    line = f.readline()
    if not line: break
    print(line)
f.close()

- ??? ?? ??? ?? ? ? ?? (??, ??, ??, ??? ?? ??, ?? ?? ?? ?? )
- ???? ?? ??? ?? ? ? ?? (??? ?? ????, ???? ??, ??)

- ???? ??? ???? ??? ??
- ??? ??? ????? ???? ??? ? ?? ???? ??? ???? ??

In [ ]:
data_path = Path(os.getenv('HOME')+'/aiffel/yoochoose/data') 
train_path = data_path / 'yoochoose-clicks.dat'
train_path

In [ ]:
def load_data(data_path: Path, nrows=None):
    data = pd.read_csv(data_path, sep=',', header=None, usecols=[0, 1, 2],
                       parse_dates=[1], dtype={0: np.int32, 2: np.int32}, nrows=nrows)
    data.columns = ['SessionId', 'Time', 'ItemId']
    return data

In [ ]:
data = load_data(train_path, None)
data.sort_values(['SessionId', 'Time'], inplace=True)  # data? id? ?? ??? ??
data

----
## 2. ??? ???

### 2.1. ???? ???? ??

In [ ]:
data['SessionId'].nunique(), data['ItemId'].nunique()

### 2.2. Session Length
`SessionId` ? ???? ??? row? ??? `session_length` ? ??

In [ ]:
session_length = data.groupby('SessionId').size()
session_length

- ??? `SessionId` ? ?????? ???? ??? ? ?? ???? ?? ?? ???
- ????? ??? ??? ? ???? ???? ?? ???, ??? ?? ???? ??? `SessionId `???? ??? ???? ?? ??
- ??? ??? `session_length` ? ?? ??? ???? ? ?? ?? ??? ??? ????(??? ????? ?????)? ??

In [ ]:
session_length.median(), session_length.mean()

In [ ]:
session_length.min(), session_length.max()

In [ ]:
session_length.quantile(0.999)

- ? ??? ??? ?? 2~3 ??
- ?? ??? 1, ?? ??? 200
- 99.9%? ??? ??? 41 ??

??? 200? ??? ?? ????? ?? ??? ????.

In [ ]:
long_session = session_length[session_length==200].index[0]
data[data['SessionId']==long_session]

?? ?? ???? ????? ??? 1?? 30? ?? ???? ??  

??? ??? ????? ??? ?????? ?????

???? ?? ?? 99.9%??? ?? ???? ??????.

In [ ]:
length_count = session_length.groupby(session_length).size()
length_percent_cumsum = length_count.cumsum() / length_count.sum()
length_percent_cumsum_999 = length_percent_cumsum[length_percent_cumsum < 0.999]

length_percent_cumsum_999

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))
plt.bar(x=length_percent_cumsum_999.index,
        height=length_percent_cumsum_999, color='red')
plt.xticks(length_percent_cumsum_999.index)
plt.yticks(np.arange(0, 1.01, 0.05))
plt.title('Cumsum Percentage Until 0.999', size=20)
plt.show()

### 2.3. Session Time

E-commerce ?? ??? ?????? ??? ??? ?? ?? ???? ???? ?? ????.

In [ ]:
oldest, latest = data['Time'].min(), data['Time'].max()
print(oldest) 
print(latest)

In [ ]:
type(latest)

`latest`? Timestamp ???? ??? int ???? ????? ???? ???.  
????? ??? ??? ?? ?? `datetime` ?????? `timedelta` ??? ????.

In [ ]:
month_ago = latest - dt.timedelta(30)     # ?? ????? 30? ?? ??? ???.  
data = data[data['Time'] > month_ago]   # ?? ?? ?? ??? ???? ???. 
data

### 2.4. Data Cleansing

- ??? ????? ??? ??? 1? ??? ???. ?? ??? ??? ??? ?? 1? ?? ???? ? ?? ??? ???? ???? ??? 1? ??? ????.  


- ?? ?? ??? ???? ??? ???? ???? ???? ????.

In [ ]:
def cleanse_short_session(data: pd.DataFrame, shortest):
    session_len = data.groupby('SessionId').size()
    session_use = session_len[session_len >= shortest].index
    data = data[data['SessionId'].isin(session_use)]
    return data


def cleanse_unpopular_item(data: pd.DataFrame, least_click):
    item_popular = data.groupby('ItemId').size()
    item_use = item_popular[item_popular >= least_click].index
    data = data[data['ItemId'].isin(item_use)]
    return data


# short_session? ??? ?? unpopular item? ???? ?? ??? 1? session? ?? ? ????.
# ?? ?? ???? ?? ????? ?? ???.
def cleanse_recursive(data: pd.DataFrame, shortest, least_click) -> pd.DataFrame:
    while True:
        before_len = len(data)
        data = cleanse_short_session(data, shortest)
        data = cleanse_unpopular_item(data, least_click)
        after_len = len(data)
        if before_len == after_len:
            break
    return data

In [ ]:
data = cleanse_recursive(data, shortest=2, least_click=5)
data

### 2.5. Train/ Valid/ Test split 
?? ??? ?? Train Set, Valid Set? Test Set? ???.  

Test Set? ???? ???? Test Set? ??? ????.

In [ ]:
test_path = data_path / 'yoochoose-test.dat'
test= load_data(test_path)
test['Time'].min(), test['Time'].max()

Test ?? ???? Training ?? ??? ???.  

?? ???? __?? ? ??__?? ? ????. 

????? ?? ??? ???? ??? 1? ?? ??? ?? ??? ?? ?? ?? ?? ?? ??.


?? ??? Session-Based Recommendation??? ??? ?? Train/ Valid/ Test ?? ???? ??.

?? ????? ?? ??? 1? ?? ??? Test?, 2???? 1?? ??? valid set?? ????.

![image](https://user-images.githubusercontent.com/84179578/130547785-d6e54244-704d-4ecc-a308-3002115aab87.png)

In [ ]:
def split_by_date(data: pd.DataFrame, n_days: int):
    final_time = data['Time'].max()
    session_last_time = data.groupby('SessionId')['Time'].max()
    session_in_train = session_last_time[session_last_time < final_time - dt.timedelta(n_days)].index
    session_in_test = session_last_time[session_last_time >= final_time - dt.timedelta(n_days)].index

    before_date = data[data['SessionId'].isin(session_in_train)]
    after_date = data[data['SessionId'].isin(session_in_test)]
    after_date = after_date[after_date['ItemId'].isin(before_date['ItemId'])]
    return before_date, after_date

In [ ]:
tr, test = split_by_date(data, n_days=1)
tr, val = split_by_date(tr, n_days=1)

?? train, val, test set ? ?? ? ??? ????.

In [ ]:
# data? ?? ??? ?????.
def stats_info(data: pd.DataFrame, status: str):
    print(f'* {status} Set Stats Info\n'
          f'\t Events: {len(data)}\n'
          f'\t Sessions: {data["SessionId"].nunique()}\n'
          f'\t Items: {data["ItemId"].nunique()}\n'
          f'\t First Time : {data["Time"].min()}\n'
          f'\t Last Time : {data["Time"].max()}\n')

In [ ]:
stats_info(tr, 'train')
stats_info(val, 'valid')
stats_info(test, 'test')

`itemid` ? ??? ??? ????.

In [ ]:
# train set? ?? ???? val, test??? ?? ? ???? train data? ???? ??????.
id2idx = {item_id : index for index, item_id in enumerate(tr['ItemId'].unique())}

def indexing(df, id2idx):
    df['item_idx'] = df['ItemId'].map(lambda x: id2idx.get(x, -1))  # id2idx? ?? ???? ??? ?(-1) ?? ????.
    return df

tr = indexing(tr, id2idx)
val = indexing(val, id2idx)
test = indexing(test, id2idx)

In [ ]:
tr.head()

???? ???? ????.

In [ ]:
save_path = data_path / 'processed'
save_path.mkdir(parents=True, exist_ok=True)

tr.to_pickle(save_path / 'train.pkl')
val.to_pickle(save_path / 'valid.pkl')
test.to_pickle(save_path / 'test.pkl')

----
## 3. Data Pipeline

Session-Parallel Mini_Batch ? ?????.

>Session-Parallel Mini-Batch ? ?? ??? ??? ?? ???? ?????.   
?? ?? ?? ??

### 3.1. SessionDataset
???? ???? ??? ???? ???? ?? ?? ??? ?? ???? ?? ?? ???? ???.

In [ ]:
class SessionDataset:
    """Credit to yhs-968/pyGRU4REC."""

    def __init__(self, data):
        self.df = data
        self.click_offsets = self.get_click_offsets()
        self.session_idx = np.arange(self.df['SessionId'].nunique())  # indexing to SessionId

    def get_click_offsets(self):
        """
        Return the indexes of the first click of each session IDs,
        """
        offsets = np.zeros(self.df['SessionId'].nunique() + 1, dtype=np.int32)
        offsets[1:] = self.df.groupby('SessionId').size().cumsum()
        return offsets

In [ ]:
tr_dataset = SessionDataset(tr)
tr_dataset.df.head(10)

In [ ]:
tr_dataset.click_offsets

In [ ]:
tr_dataset.session_idx

train???? `SessionDataset` ??? ??? ???? ???? ??? ??.

- `click_offsets` ??? ? ??? ??? ???? ?? ??.
- `session_idx` ??? ? ??? ???? np.array ??.

### 3.2. SessionDataLoader

`SessionDataset` ??? ??? Session-Parallel mini-batch? ??? ???? ???.  

`__iter__` ???? ?? ??, ??, ??? ??? ?? ??? `yield` ??.  
`mask`? ??? RNN Cell State? ??? ??? ??? ???.

In [ ]:
class SessionDataLoader:
    """Credit to yhs-968/pyGRU4REC."""

    def __init__(self, dataset: SessionDataset, batch_size=50):
        self.dataset = dataset
        self.batch_size = batch_size

    def __iter__(self):
        """ Returns the iterator for producing session-parallel training mini-batches.
        Yields:
            input (B,):  Item indices that will be encoded as one-hot vectors later.
            target (B,): a Variable that stores the target item indices
            masks: Numpy array indicating the positions of the sessions to be terminated
        """

        start, end, mask, last_session, finished = self.initialize()  # initialize ????? ??????.
        """
        start : Index Where Session Start
        end : Index Where Session End
        mask : indicator for the sessions to be terminated
        """

        while not finished:
            min_len = (end - start).min() - 1  # Shortest Length Among Sessions
            for i in range(min_len):
                # Build inputs & targets
                inp = self.dataset.df['item_idx'].values[start + i]
                target = self.dataset.df['item_idx'].values[start + i + 1]
                yield inp, target, mask

            start, end, mask, last_session, finished = self.update_status(start, end, min_len, last_session, finished)

    def initialize(self):
        first_iters = np.arange(self.batch_size)    # ? ??? ??? ?? Index? ?????.
        last_session = self.batch_size - 1    # ????? ??? ?? ?? Index? ??????.
        start = self.dataset.click_offsets[self.dataset.session_idx[first_iters]]       # data ??? session? ??? ??? ?????.
        end = self.dataset.click_offsets[self.dataset.session_idx[first_iters] + 1]  # session? ?? ?? ?? ?? ??? ?????.
        mask = np.array([])   # session? ?? ???? ? ?? ?? mask? ???? ????.
        finished = False         # data? ?? ???? ???? ?? ?????.
        return start, end, mask, last_session, finished

    def update_status(self, start: np.ndarray, end: np.ndarray, min_len: int, last_session: int, finished: bool):  
        # ?? ?? ???? ???? ?? ??? update???.
        
        start += min_len   # __iter__?? min_len ?? for?? ????? start? min_len ?? ?????.
        mask = np.arange(self.batch_size)[(end - start) == 1]  
        # end? ?? ??? ???? ???? start? ? ? ????? ?? session? ???? ????. mask? ??????.

        for i, idx in enumerate(mask, start=1):  # mask? ??? ?? ???? ??? ??? ?????.
            new_session = last_session + i  
            if new_session > self.dataset.session_idx[-1]:  # ?? ??? ??? ??? ?? index?? ??? ?? ?????? ? ????.
                finished = True
                break
            # update the next starting/ending point
            start[idx] = self.dataset.click_offsets[self.dataset.session_idx[new_session]]     # ??? ?? ?? ??? ??? ???? ?????.
            end[idx] = self.dataset.click_offsets[self.dataset.session_idx[new_session] + 1]

        last_session += len(mask)  # ??? ??? ??? ??????.
        return start, end, mask, last_session, finished

In [ ]:
tr_data_loader = SessionDataLoader(tr_dataset, batch_size=4)
tr_dataset.df.head(15)

In [ ]:
iter_ex = iter(tr_data_loader)

?? ?? ?? ???? `next`? ?? ?? ???? ???.

In [ ]:
inputs, labels, mask =  next(iter_ex)
print(f'Model Input Item Idx are : {inputs}')
print(f'Label Item Idx are : {"":5} {labels}')
print(f'Previous Masked Input Idx are {mask}')

In [ ]:
inputs, labels, mask =  next(iter_ex)
print(f'Model Input Item Idx are : {inputs}')
print(f'Label Item Idx are : {"":5} {labels}')
print(f'Previous Masked Input Idx are {mask}')

In [ ]:
inputs, labels, mask =  next(iter_ex)
print(f'Model Input Item Idx are : {inputs}')
print(f'Label Item Idx are : {"":5} {labels}')
print(f'Previous Masked Input Idx are {mask}')

In [ ]:
inputs, labels, mask =  next(iter_ex)
print(f'Model Input Item Idx are : {inputs}')
print(f'Label Item Idx are : {"":5} {labels}')
print(f'Previous Masked Input Idx are {mask}')

----
## 4. Modeling
### 4.1. Evaluation Metric

?? ??? ?? ??? __MRR__ ? __Recall@k__ (recall? ??? ??) ? ?????.  

MMR ? ?? ???? ?? ??? ?? ???. ??? ?? ???? ?? ?? ?? ??? ???? ??? ??? ??? ? ?? ???? ????? ??? ??? ???.  

Session-Based Recommendation Task ?? ???? ??? ?? ?? ?? ??(NDCG, MRR, MAP ?)? ?? ?? ???? ?????.  

> ??? ??

In [ ]:
def mrr_k(pred, truth: int, k: int):
    indexing = np.where(pred[:k] == truth)[0]
    if len(indexing) > 0:
        return 1 / (indexing[0] + 1)
    else:
        return 0


def recall_k(pred, truth: int, k: int) -> int:
    answer = truth in pred[:k]
    return int(answer)

### 4.2. Model Architecture

2016? ICLR? ??? [SESSION-BASED RECOMMENDATIONS WITH RECURRENT NEURAL NETWORKS](https://arxiv.org/pdf/1511.06939v4.pdf) ?? ??? Functional ? ???? ?????.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, GRU
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

In [ ]:
def create_model(args):
    inputs = Input(batch_shape=(args.batch_size, 1, args.num_items))
    gru, _ = GRU(args.hsz, stateful=True, return_state=True, name='GRU')(inputs)
    dropout = Dropout(args.drop_rate)(gru)
    predictions = Dense(args.num_items, activation='softmax')(dropout)
    model = Model(inputs=inputs, outputs=[predictions])
    model.compile(loss=categorical_crossentropy, optimizer=Adam(args.lr), metrics=['accuracy'])
    model.summary()
    return model

??? ??? hyper-parameter ? class ???? ?????.

In [ ]:
class Args:
    def __init__(self, tr, val, test, batch_size, hsz, drop_rate, lr, epochs, k):
        self.tr = tr
        self.val = val
        self.test = test
        self.num_items = tr['ItemId'].nunique()
        self.num_sessions = tr['SessionId'].nunique()
        self.batch_size = batch_size
        self.hsz = hsz
        self.drop_rate = drop_rate
        self.lr = lr
        self.epochs = epochs
        self.k = k

args = Args(tr, val, test, batch_size=2048, hsz=50, drop_rate=0.1, lr=0.001, epochs=3, k=20)

In [ ]:
model = create_model(args)

----
## 5. Model Training

??? ? 3epoch ????.  
3epoch? ??? ??? ???? ??? ??? ??? ???, ?? ??? ????? ???? ??? ??? ? ?? ??? ? ???.

In [ ]:
# train ??? ????? valid ??? ?????.
def train_model(model, args):
    train_dataset = SessionDataset(args.tr)
    train_loader = SessionDataLoader(train_dataset, batch_size=args.batch_size)

    for epoch in range(1, args.epochs + 1):
        total_step = len(args.tr) - args.tr['SessionId'].nunique()
        tr_loader = tqdm(train_loader, total=total_step // args.batch_size, desc='Train', mininterval=1)
        for feat, target, mask in tr_loader:
            reset_hidden_states(model, mask)  # ??? session? hidden_state? ??????. ?? ????? ??????.

            input_ohe = to_categorical(feat, num_classes=args.num_items)
            input_ohe = np.expand_dims(input_ohe, axis=1)
            target_ohe = to_categorical(target, num_classes=args.num_items)

            result = model.train_on_batch(input_ohe, target_ohe)
            tr_loader.set_postfix(train_loss=result[0], accuracy = result[1])

        val_recall, val_mrr = get_metrics(args.val, model, args, args.k)  # valid set? ?? ?????.

        print(f"\t - Recall@{args.k} epoch {epoch}: {val_recall:3f}")
        print(f"\t - MRR@{args.k}    epoch {epoch}: {val_mrr:3f}\n")


def reset_hidden_states(model, mask):
    gru_layer = model.get_layer(name='GRU')  # model?? gru layer? ?????.
    hidden_states = gru_layer.states[0].numpy()  # gru_layer? parameter? ?????.
    for elt in mask:  # mask? ??? ?, ??? ??? ???? ???
        hidden_states[elt, :] = 0  # parameter? ??? ???.
    gru_layer.reset_states(states=hidden_states)


def get_metrics(data, model, args, k: int):  # valid?? test?? ???? ?????. 
                                             # train? ?? ??? mrr, recall? ??? ??? ????.
    dataset = SessionDataset(data)
    loader = SessionDataLoader(dataset, batch_size=args.batch_size)
    recall_list, mrr_list = [], []

    total_step = len(data) - data['SessionId'].nunique()
    for inputs, label, mask in tqdm(loader, total=total_step // args.batch_size, desc='Evaluation', mininterval=1):
        reset_hidden_states(model, mask)
        input_ohe = to_categorical(inputs, num_classes=args.num_items)
        input_ohe = np.expand_dims(input_ohe, axis=1)

        pred = model.predict(input_ohe, batch_size=args.batch_size)
        pred_arg = tf.argsort(pred, direction='DESCENDING')  # softmax ?? ? ???? sorting ???.

        length = len(inputs)
        recall_list.extend([recall_k(pred_arg[i], label[i], k) for i in range(length)])
        mrr_list.extend([mrr_k(pred_arg[i], label[i], k) for i in range(length)])

    recall, mrr = np.mean(recall_list), np.mean(mrr_list)
    return recall, mrr

In [ ]:
train_model(model, args)

----
## 6. Model Test 

??? ??? ????? ???? ??? ??? ??? ??? ?????.

In [ ]:
def test_model(model, args, test):
    test_recall, test_mrr = get_metrics(test, model, args, 20)
    print(f"\t - Recall@{args.k}: {test_recall:3f}")
    print(f"\t - MRR@{args.k}: {test_mrr:3f}\n")

test_model(model, args, test)

test set ??? ??? ??? ??? ??? ?? ?? ? ? ??.  

???? ??? epoch ? 3?? ???? ?? ???? ??? ????, ?? epoch ? ???? ??? ???? ??? ? ?? ??? ?? ??? ????. 